## Usernames filter

This notebook aims to filter the raw usernames contained in `usernames.txt`, to keep only those created between March, 17th and September, 17th, and who let public their production.

### Chose subset of the total `usernames.txt`

In [1]:
X = 3 # Or 2 or 3

In [2]:
with open('usernames.txt', 'r', encoding='utf-8') as f:
    raw_names = f.readlines()

breakpoint = int(len(raw_names)/3)
batch = raw_names[breakpoint*(X-1):breakpoint*(X)]
if X == 3:
    batch += raw_names[breakpoint*(X)+1:]

# Open already scraped names to avoid rescraping them
try : 
    with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
        last_done = f.readlines()[-1]
        if "Last" in last_done:
            last_done = last_done.split(": ", 1)[1]
            batch = batch[batch.index(last_done)+1 : ]
        else : 
            print('No past attempt saved.')
        already_file = True
except Exception:
    print('No previous file found.')
    already_file = False

print(f'Remaining length batch: {len(batch)}')

No previous file found.
Remaining length batch: 33327


### Scrap Reddit to filter each username in the batch

In [3]:
import requests
from datetime import datetime
import time
import random 
import string

start = datetime(2020, 3, 17)
end = datetime(2020, 9, 17)

to_save = []
missing = 0
headuser = "Mozilla/5.0 (compatible; scraper/1.0)"

for i, username in enumerate(batch):
    if i % 30 ==0:
        time.sleep(15)
        headuser += random.choice(string.ascii_uppercase + string.digits)
        headers = {
            "User-Agent": headuser
        }
        session = requests.Session()
        session.headers.update(headers)
    if i%20==0:
        print(f'Step {i}')
    time.sleep(0.5)
    try:
        r = session.get(
            f"https://www.reddit.com/user/{username.strip('\n')}/about.json", 
            timeout=3)
        r.raise_for_status()
        if str(r.status_code) in ["429", "403", "401"]:
            print("Blocked:", r.status_code)
            raise Exception("Hard block")

        if "application/json" not in r.headers.get("Content-Type", ""):
            print("Soft block (not JSON)")
            print(r.text[:200])
            raise Exception("Soft block")

        created_utc = r.json()["data"]["created_utc"]
    except Exception:
        missing+=1
        if missing % 5 == 0: 
            percent = (missing * 100) / (i + 1)
            print(f"Missing: {percent:.2f}%")
        continue
    date_regis = datetime.utcfromtimestamp(created_utc)
    if start <= date_regis <= end:
        r = session.get(
            f"https://www.reddit.com/user/{username.strip('\n')}/.json",
            timeout=3)
        r.raise_for_status()
        data = r.json()
        if data['data']['children']:
            to_save.append(username)
            print(f'Found: {len(to_save)}, Among: {i+1}')

print(f'Missing:{missing}')

with open(f'covid_users_{X}.txt', 'w', encoding='utf-8') as f:
    for username in to_save:
        f.write(username)

Step 0


/tmp/ipykernel_1029/2298169990.py:47: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date_regis = datetime.utcfromtimestamp(created_utc)


Found: 1, Among: 5
Found: 2, Among: 7
Found: 3, Among: 11
Missing: 35.71%
Step 20
Found: 4, Among: 32
Missing: 26.32%
Step 40
Missing: 26.79%
Found: 5, Among: 58
Step 60
Found: 6, Among: 71
Missing: 25.64%
Step 80
Found: 7, Among: 94
Step 100
Step 120
Missing: 20.49%
Missing: 21.90%
Step 140
Step 160
Missing: 20.59%
Found: 8, Among: 176
Step 180
Found: 9, Among: 185
Found: 10, Among: 187
Found: 11, Among: 193
Missing: 20.41%
Found: 12, Among: 198
Step 200
Found: 13, Among: 206
Step 220
Found: 14, Among: 230
Missing: 18.83%
Step 240
Missing: 19.53%
Step 260
Missing: 20.15%
Found: 15, Among: 276
Found: 16, Among: 277
Step 280
Found: 17, Among: 284
Missing: 20.07%
Step 300
Step 320
Missing: 20.00%
Found: 18, Among: 326
Step 340
Missing: 20.47%
Missing: 21.25%
Step 360
Missing: 21.80%
Missing: 22.43%
Step 380
Found: 19, Among: 389
Step 400
Missing: 22.39%
Missing: 22.67%
Step 420
Found: 20, Among: 432
Step 440
Missing: 22.12%
Step 460
Found: 21, Among: 462
Found: 22, Among: 468
Found: 23, 

KeyboardInterrupt: 

### Temporary save

Run the cell below to save your progression in case the previous cell raised an error, or you keyboard interrupted it. 

In [4]:
if already_file:
    with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
        found = f.readlines()
        if "NOT" in found[-1]:
            to_save = found[-2::-1] + to_save
        else : 
            found[-1] = found[-1].strip("Last (found): ")
            to_save = found + to_save

with open(f'covid_users_{X}.txt', 'w', encoding='utf-8') as f:
    for user in to_save:
        f.write(user)
    if username in to_save:
        f.write(f'Last (found): {username}')
    else : 
        f.write(f'Last (NOT found): {username}')